# Ears, Brain, Mouth: Controlled AI Pipeline Experiments

This notebook isolates the three stages of your project:

1. **Ears** — transcribe one recorded tricky question with `faster-whisper`
2. **Brain** — send that transcript to multiple models and system prompts repeatedly
3. **Mouth** — synthesize one answer with Coqui XTTS and judge the cloned voice qualitatively

This version removes Discord entirely and follows the same audio preprocessing pattern used in `handle_stop` from your script:
raw PCM bytes -> NumPy samples -> float32 normalization -> mono fold -> resample to 16 kHz -> `transcribe_audio_memory(...)`

The **Brain** stage now manages `llama-server` directly:
- start one model server
- wait for `/health`
- run all trials for that model
- stop the server
- repeat for the next model


## 0. Imports and setup

In [1]:
import asyncio
import difflib
import gc
import json
import logging
import os
import re
import subprocess
import time
import wave
from pathlib import Path

import aiohttp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.signal
from IPython.display import Audio, display
from dotenv import load_dotenv
from faster_whisper import WhisperModel
from TTS.api import TTS

logging.basicConfig(level=logging.INFO)
load_dotenv()

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)


## 1. Configuration

In [ ]:
AUDIO_INPUT_PATH = Path(os.getenv("AUDIO_INPUT_PATH", "input_question.wav"))
GOLD_TEXT = "I need to wash my car. The car wash is only 100 meters away. Should I walk or drive?"

WHISPER_MODEL_SIZE = os.getenv("WHISPER_MODEL", "base")
WHISPER_DEVICE = os.getenv("WHISPER_DEVICE", "cpu")
WHISPER_COMPUTE_TYPE = os.getenv("WHISPER_COMPUTE_TYPE", "int8")

MANAGE_LLAMA_SERVER = True
LLAMA_SERVER_HOST = os.getenv("LLAMA_SERVER_HOST", "127.0.0.1")
LLAMA_SERVER_PORT = int(os.getenv("LLAMA_SERVER_PORT", "8080"))
LLAMA_SERVER_BASE_URL = f"http://{LLAMA_SERVER_HOST}:{LLAMA_SERVER_PORT}"
LLAMA_SERVER_URL = f"{LLAMA_SERVER_BASE_URL}/v1/chat/completions"

LLAMA_CONNECT_TIMEOUT = float(os.getenv("LLAMA_CONNECT_TIMEOUT", "10"))
LLAMA_SOCK_READ_TIMEOUT = float(os.getenv("LLAMA_SOCK_READ_TIMEOUT", "300"))
LLAMA_SERVER_STARTUP_TIMEOUT = float(os.getenv("LLAMA_SERVER_STARTUP_TIMEOUT", "600"))
LLAMA_SERVER_SHUTDOWN_TIMEOUT = float(os.getenv("LLAMA_SERVER_SHUTDOWN_TIMEOUT", "20"))
LLAMA_MAX_TOKENS = int(os.getenv("LLAMA_MAX_TOKENS", "80"))
LLAMA_TEMPERATURE = float(os.getenv("LLAMA_TEMPERATURE", "0.8"))
LLAMA_RETRIES = int(os.getenv("LLAMA_RETRIES", "2"))
LLAMA_RETRY_BACKOFF_S = float(os.getenv("LLAMA_RETRY_BACKOFF_S", "2"))

MODEL_CONFIGS = [
    {
        "label": "Small model",
        "model": "unsloth/Qwen3.5-9B-GGUF:UD-Q6_K_XL",
        "server_cmd": [
            "llama-server",
            "-hf", "unsloth/Qwen3.5-9B-GGUF:UD-Q6_K_XL",
            "--no-mmproj",
            "-c", "8192",
            "--reasoning-budget", "0.5",
            "--host", LLAMA_SERVER_HOST,
            "--port", str(LLAMA_SERVER_PORT),
        ],
    },
    {
        "label": "Large model",
        "model": "unsloth/Qwen3.6-35B-A3B-GGUF:UD-Q4_K_M",
        "server_cmd": [
            "llama-server",
            "-hf", "unsloth/Qwen3.6-35B-A3B-GGUF:UD-Q4_K_M",
            "--fit-ctx", "8192",
            "-np", "1",
            "-fa", "on",
            "-b", "512",
            "-ub", "512",
            "-ctk", "q8_0",
            "-ctv", "q8_0",
            "--temp", "0.6",
            "--top-p", "0.95",
            "--top-k", "20",
            "--min-p", "0.0",
            "--presence-penalty", "0.0",
            "--repeat-penalty", "1.0",
            "--reasoning-budget", "-1",
            "--chat-template-kwargs", '{"preserve_thinking": true}',
            "--host", LLAMA_SERVER_HOST,
            "--port", str(LLAMA_SERVER_PORT),
        ],
    },
]

PROMPTS = {
    "baseline": "Answer in one sentence.",
    "novice_literal": "In one sentence, answer like a novice who interprets the question literally and simply.",
    "phd_student": "In one sentence, answer like a careful PhD student who reasons step by step and notices subtle assumptions.",
    "trick_specialist": "In one sentence, answer like a specialist at detecting tricky questions, hidden assumptions, wordplay, and catches. If there is a trick, explain it clearly.",
}

N_RUNS = 50

COQUI_MODEL_NAME = os.getenv("COQUI_MODEL", "tts_models/multilingual/multi-dataset/xtts_v2")
COQUI_LANGUAGE = os.getenv("COQUI_LANGUAGE", "en")
COQUI_SPEAKER_WAV = os.getenv("COQUI_SPEAKER_WAV", "my_voice_reference.wav")

print("AUDIO_INPUT_PATH:", AUDIO_INPUT_PATH)
print("Exists:", AUDIO_INPUT_PATH.exists())
print("MANAGE_LLAMA_SERVER:", MANAGE_LLAMA_SERVER)
print("LLAMA_SERVER_BASE_URL:", LLAMA_SERVER_BASE_URL)
print("Small model command:", " ".join(MODEL_CONFIGS[0]["server_cmd"]))
print("Large model command:", " ".join(MODEL_CONFIGS[1]["server_cmd"]))


AUDIO_INPUT_PATH: /home/havoc/git_repos/Discord_AI_bot/src/outputs/input_question.wav
Exists: True
MANAGE_LLAMA_SERVER: True
LLAMA_SERVER_BASE_URL: http://127.0.0.1:8080
Small model command: llama-server -hf unsloth/Qwen3.5-9B-GGUF:UD-Q6_K_XL --no-mmproj -c 8192 --reasoning-budget 0.5 --host 127.0.0.1 --port 8080
Large model command: llama-server -hf unsloth/Qwen3.6-35B-A3B-GGUF:UD-Q4_K_M --fit-ctx 8192 -np 1 -fa on -b 512 -ub 512 -ctk q8_0 -ctv q8_0 --temp 0.6 --top-p 0.95 --top-k 20 --min-p 0.0 --presence-penalty 0.0 --repeat-penalty 1.0 --reasoning-budget -1 --chat-template-kwargs {"preserve_thinking": true} --host 127.0.0.1 --port 8080


## 2. Shared helpers

In [3]:
stt_model = None
tts = None
llama_server_process = None
llama_server_log_handle = None
llama_server_log_path = None

def load_stt_model():
    global stt_model
    if stt_model is None:
        logging.info("Loading faster-whisper model: %s", WHISPER_MODEL_SIZE)
        stt_model = WhisperModel(
            WHISPER_MODEL_SIZE,
            device=WHISPER_DEVICE,
            compute_type=WHISPER_COMPUTE_TYPE,
        )
    return stt_model

def unload_stt_model():
    global stt_model
    stt_model = None
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass

def load_tts_model():
    global tts
    if tts is None:
        logging.info("Loading Coqui TTS model: %s", COQUI_MODEL_NAME)
        tts = TTS(model_name=COQUI_MODEL_NAME)
        try:
            tts.to("cuda")
            logging.info("Moved Coqui TTS to CUDA")
        except Exception:
            logging.info("Using default device for Coqui TTS")
    return tts

def _pcm24_to_int32(raw_bytes):
    data = np.frombuffer(raw_bytes, dtype=np.uint8)
    if len(data) % 3 != 0:
        raise ValueError("24-bit PCM data length is not divisible by 3.")
    data = data.reshape(-1, 3)
    out = (
        data[:, 0].astype(np.int32)
        | (data[:, 1].astype(np.int32) << 8)
        | (data[:, 2].astype(np.int32) << 16)
    )
    sign_mask = 1 << 23
    out = np.where(out & sign_mask, out - (1 << 24), out)
    return out.astype(np.int32)

def read_wav_like_handle_stop(audio_path):
    audio_path = Path(audio_path)

    if not audio_path.exists():
        raise FileNotFoundError(f"Audio file not found: {audio_path}")
    if audio_path.stat().st_size == 0:
        raise ValueError(f"Audio file is empty: {audio_path}")

    with wave.open(str(audio_path), "rb") as wf:
        channels = wf.getnchannels()
        rate = wf.getframerate()
        sample_width = wf.getsampwidth()
        n_frames = wf.getnframes()
        raw_data = wf.readframes(n_frames)

    if sample_width == 1:
        audio_np = np.frombuffer(raw_data, dtype=np.uint8).astype(np.float32)
        audio_float32 = (audio_np - 128.0) / 128.0
    elif sample_width == 2:
        audio_np = np.frombuffer(raw_data, dtype=np.int16)
        audio_float32 = audio_np.astype(np.float32) / 32768.0
    elif sample_width == 3:
        audio_np = _pcm24_to_int32(raw_data)
        audio_float32 = audio_np.astype(np.float32) / 8388608.0
    elif sample_width == 4:
        audio_np = np.frombuffer(raw_data, dtype=np.int32)
        audio_float32 = audio_np.astype(np.float32) / 2147483648.0
    else:
        raise ValueError(f"Unsupported PCM sample width: {sample_width} bytes")

    if channels > 1:
        audio_float32 = audio_float32.reshape(-1, channels).mean(axis=1)

    original_rate = rate
    target_rate = 16000
    if original_rate != target_rate:
        num_samples = int(len(audio_float32) * float(target_rate) / original_rate)
        audio_16k = scipy.signal.resample(audio_float32, num_samples).astype(np.float32)
    else:
        audio_16k = audio_float32.astype(np.float32)

    return {
        "rate": original_rate,
        "channels": channels,
        "sample_width": sample_width,
        "n_frames": n_frames,
        "audio_16k": audio_16k,
        "duration_s": len(audio_float32) / float(original_rate),
    }

async def transcribe_audio_memory(audio_data: np.ndarray, language="en", vad_filter=True):
    model = load_stt_model()
    segments, info = await asyncio.to_thread(
        model.transcribe,
        np.asarray(audio_data, dtype=np.float32),
        language=language,
        vad_filter=vad_filter,
    )
    text = "".join(segment.text for segment in segments).strip()
    return text, info

def normalize_text(text):
    return " ".join(text.lower().strip().split())

def transcript_similarity(reference, hypothesis):
    ref = normalize_text(reference)
    hyp = normalize_text(hypothesis)
    return difflib.SequenceMatcher(None, ref, hyp).ratio()

async def transcribe_wav(audio_path, language="en", vad_filter=True):
    prep = read_wav_like_handle_stop(audio_path)
    start = time.perf_counter()
    transcript, info = await transcribe_audio_memory(
        prep["audio_16k"],
        language=language,
        vad_filter=vad_filter,
    )
    elapsed = time.perf_counter() - start
    return {
        "audio_path": str(audio_path),
        "language": language,
        "transcript": transcript,
        "elapsed_s": elapsed,
        "duration_s": prep["duration_s"],
        "detected_language": getattr(info, "language", None),
        "language_probability": getattr(info, "language_probability", None),
        "source_rate": prep["rate"],
        "source_channels": prep["channels"],
        "source_sample_width": prep["sample_width"],
        "source_frames": prep["n_frames"],
    }

def make_llama_timeout():
    return aiohttp.ClientTimeout(
        total=None,
        connect=LLAMA_CONNECT_TIMEOUT,
        sock_connect=LLAMA_CONNECT_TIMEOUT,
        sock_read=LLAMA_SOCK_READ_TIMEOUT,
    )

def _safe_name(text):
    return re.sub(r"[^A-Za-z0-9._-]+", "_", str(text)).strip("_") or "model"

async def wait_for_llama_server_ready(base_url=None, timeout_s=None):
    base_url = base_url or LLAMA_SERVER_BASE_URL
    timeout_s = timeout_s or LLAMA_SERVER_STARTUP_TIMEOUT
    health_url = f"{base_url}/health"
    deadline = time.monotonic() + timeout_s

    while time.monotonic() < deadline:
        global llama_server_process
        if llama_server_process is not None and llama_server_process.poll() is not None:
            raise RuntimeError(f"llama-server exited early with code {llama_server_process.returncode}")

        try:
            timeout = aiohttp.ClientTimeout(total=5, connect=3, sock_connect=3, sock_read=3)
            async with aiohttp.ClientSession(timeout=timeout) as session:
                async with session.get(health_url) as response:
                    if response.status == 200:
                        return True
        except Exception:
            pass

        await asyncio.sleep(1)

    raise TimeoutError(f"Timed out waiting for llama-server health at {health_url}")

async def stop_llama_server():
    global llama_server_process, llama_server_log_handle, llama_server_log_path

    if llama_server_process is None:
        return

    proc = llama_server_process
    try:
        if proc.poll() is None:
            proc.terminate()
            try:
                await asyncio.to_thread(proc.wait, timeout=LLAMA_SERVER_SHUTDOWN_TIMEOUT)
            except subprocess.TimeoutExpired:
                proc.kill()
                await asyncio.to_thread(proc.wait, timeout=5)
    finally:
        if llama_server_log_handle is not None:
            llama_server_log_handle.close()
        llama_server_process = None
        llama_server_log_handle = None
        llama_server_log_path = None

async def start_llama_server(model_cfg):
    global llama_server_process, llama_server_log_handle, llama_server_log_path

    await stop_llama_server()

    cmd = model_cfg["server_cmd"]
    label = _safe_name(model_cfg["label"])
    llama_server_log_path = OUTPUT_DIR / f"llama_server_{label}.log"
    llama_server_log_handle = open(llama_server_log_path, "w", encoding="utf-8", buffering=1)

    logging.info("Starting llama-server for %s", model_cfg["label"])
    logging.info("Command: %s", " ".join(cmd))

    try:
        llama_server_process = subprocess.Popen(
            cmd,
            stdout=llama_server_log_handle,
            stderr=subprocess.STDOUT,
            text=True,
            env=os.environ.copy(),
        )
    except FileNotFoundError as e:
        if llama_server_log_handle is not None:
            llama_server_log_handle.close()
        llama_server_log_handle = None
        llama_server_log_path = None
        llama_server_process = None
        raise RuntimeError("Could not find `llama-server` on PATH. Start Jupyter from the same environment where llama-server works.") from e

    await wait_for_llama_server_ready()
    return {
        "pid": llama_server_process.pid,
        "log_path": str(llama_server_log_path),
    }

async def generate_reply(user_text, model_name, system_prompt="", temperature=None, max_tokens=None, server_url=None, session=None):
    payload = {
        "model": model_name,
        "messages": [],
        "max_tokens": LLAMA_MAX_TOKENS if max_tokens is None else max_tokens,
        "temperature": LLAMA_TEMPERATURE if temperature is None else temperature,
    }

    if system_prompt:
        payload["messages"].append({"role": "system", "content": system_prompt})
    payload["messages"].append({"role": "user", "content": user_text})

    headers = {
        "Content-Type": "application/json",
        "Authorization": "Bearer no-key",
    }

    owns_session = session is None
    if owns_session:
        session = aiohttp.ClientSession(timeout=make_llama_timeout(), headers=headers)

    start = time.perf_counter()
    try:
        async with session.post(server_url or LLAMA_SERVER_URL, json=payload) as response:
            response.raise_for_status()
            data = await response.json()
            text = data["choices"][0]["message"]["content"].strip()
        elapsed = time.perf_counter() - start
        return {
            "ok": True,
            "model": model_name,
            "system_prompt": system_prompt,
            "reply": text,
            "elapsed_s": elapsed,
            "error_type": None,
            "error_message": None,
        }
    except Exception as e:
        elapsed = time.perf_counter() - start
        return {
            "ok": False,
            "model": model_name,
            "system_prompt": system_prompt,
            "reply": "",
            "elapsed_s": elapsed,
            "error_type": type(e).__name__,
            "error_message": str(e),
        }
    finally:
        if owns_session:
            await session.close()

async def generate_reply_with_retry(user_text, model_name, system_prompt="", temperature=None, max_tokens=None, server_url=None, session=None, retries=2, backoff_s=2):
    for attempt in range(retries + 1):
        result = await generate_reply(
            user_text=user_text,
            model_name=model_name,
            system_prompt=system_prompt,
            temperature=temperature,
            max_tokens=max_tokens,
            server_url=server_url,
            session=session,
        )
        if result["ok"]:
            result["attempt_count"] = attempt + 1
            return result

        retryable = result["error_type"] in {"TimeoutError", "ClientOSError", "ServerDisconnectedError", "ClientPayloadError"}
        if attempt < retries and retryable:
            sleep_s = backoff_s * (attempt + 1)
            print(f"Retrying {model_name} after {result['error_type']} in {sleep_s:.1f}s")
            await asyncio.sleep(sleep_s)
        else:
            result["attempt_count"] = attempt + 1
            return result

def label_reply(reply_text, ok=True, error_type=None):
    if not ok:
        return "borked"

    text = normalize_text(reply_text)
    if "driving is better" in text or "drive is better" in text or "driving is faster" in text or "you should drive" in text or "driving is more efficient" in text or "driving is more practical" in text:
        return "pass"
    if "walking is better" in text or "walk is better" in text or "walking is faster" in text or "you should walk" in text or "walking is more efficient" in text or "walking is more practical" in text:
        return "failed"
    return "unrelated"


## 3. Ears

Inspect the WAV metadata first, then run a single transcription pass.
Since faster-whisper turned out to be deterministic for your setup, there is no repeated STT loop anymore.


In [4]:
wav_info = read_wav_like_handle_stop(AUDIO_INPUT_PATH)
{
    "source_rate": wav_info["rate"],
    "source_channels": wav_info["channels"],
    "source_sample_width_bytes": wav_info["sample_width"],
    "source_frames": wav_info["n_frames"],
    "duration_s": wav_info["duration_s"],
    "prepared_num_samples_16k": len(wav_info["audio_16k"]),
}


{'source_rate': 44100,
 'source_channels': 2,
 'source_sample_width_bytes': 2,
 'source_frames': 357415,
 'duration_s': 8.104648526077098,
 'prepared_num_samples_16k': 129674}

In [5]:
ear_result = await transcribe_wav(AUDIO_INPUT_PATH)
transcript = ear_result["transcript"]

print("Transcript:")
print(transcript)
print()
print("Detected language:", ear_result["detected_language"])
print("Language probability:", ear_result["language_probability"])
print("Elapsed (s):", round(ear_result["elapsed_s"], 3))
print("Approx duration (s):", round(ear_result["duration_s"], 3))

# Unload Faster-Whisper before the Brain stage.
unload_stt_model()
print("Faster-Whisper model unloaded.")


INFO:root:Loading faster-whisper model: base
INFO:faster_whisper:Processing audio with duration 00:08.105
INFO:faster_whisper:VAD filter removed 00:00.144 of audio


Transcript:
I need to wash my car. The car wash is only 100 meters away. Should I walk or drive?

Detected language: en
Language probability: 1
Elapsed (s): 0.563
Approx duration (s): 8.105
Faster-Whisper model unloaded.


In [6]:
ear_metrics = {
    "audio_path": str(AUDIO_INPUT_PATH),
    "transcript": transcript,
    "gold_text": GOLD_TEXT,
    "similarity_to_gold": transcript_similarity(GOLD_TEXT, transcript),
    "elapsed_s": ear_result["elapsed_s"],
    "duration_s": ear_result["duration_s"],
    "source_rate": ear_result["source_rate"],
    "source_channels": ear_result["source_channels"],
    "source_sample_width": ear_result["source_sample_width"],
}
pd.DataFrame([ear_metrics])


,audio_path,transcript,gold_text,similarity_to_gold,elapsed_s,duration_s,source_rate,source_channels,source_sample_width
0,/home/havoc/git_repos/Discord_AI_bot/src/outpu...,I need to wash my car. The car wash is only 10...,I need to wash my car. The car wash is only 10...,1.0,0.562501,8.104649,44100,2,2


In [7]:
brain_input_text = transcript

print("Transcript chosen for Brain:")
print(brain_input_text)


Transcript chosen for Brain:
I need to wash my car. The car wash is only 100 meters away. Should I walk or drive?


## 4. Brain

Use the single transcript from the **Ears** stage as the fixed input for all model/prompt trials.

For each model, the notebook will:
1. start `llama-server` with that model's command
2. wait for the `/health` endpoint
3. run all prompt trials for that model
4. terminate the server


In [8]:
async def run_brain_experiment(user_text, model_configs, prompts, n_runs=10):
    rows = []
    total = len(model_configs) * len(prompts) * n_runs
    done = 0

    for model_cfg in model_configs:
        startup_info = None
        model_log_path = None

        if MANAGE_LLAMA_SERVER:
            try:
                startup_info = await start_llama_server(model_cfg)
                model_log_path = startup_info["log_path"]
                print(f"Started {model_cfg['label']} | pid={startup_info['pid']} | log={model_log_path}")
            except Exception as e:
                print(f"Failed to start {model_cfg['label']}: {e}")
                for prompt_name in prompts:
                    for run_idx in range(n_runs):
                        rows.append({
                            "brain_input_text": user_text,
                            "model_label": model_cfg["label"],
                            "model_name": model_cfg["model"],
                            "prompt_name": prompt_name,
                            "run_idx": run_idx + 1,
                            "ok": False,
                            "attempt_count": 0,
                            "reply": "",
                            "label_auto": "borked",
                            "elapsed_s": 0.0,
                            "error_type": type(e).__name__,
                            "error_message": str(e),
                            "server_log_path": model_log_path,
                        })
                        done += 1
                continue

        timeout = make_llama_timeout()
        headers = {
            "Content-Type": "application/json",
            "Authorization": "Bearer no-key",
        }

        try:
            async with aiohttp.ClientSession(timeout=timeout, headers=headers) as session:
                for prompt_name, system_prompt in prompts.items():
                    for run_idx in range(n_runs):
                        result = await generate_reply_with_retry(
                            user_text=user_text,
                            model_name=model_cfg["model"],
                            system_prompt=system_prompt,
                            session=session,
                            retries=LLAMA_RETRIES,
                            backoff_s=LLAMA_RETRY_BACKOFF_S,
                        )
                        rows.append({
                            "brain_input_text": user_text,
                            "model_label": model_cfg["label"],
                            "model_name": model_cfg["model"],
                            "prompt_name": prompt_name,
                            "run_idx": run_idx + 1,
                            "ok": result["ok"],
                            "attempt_count": result.get("attempt_count"),
                            "reply": result["reply"],
                            "label_auto": label_reply(result["reply"], ok=result["ok"], error_type=result["error_type"]),
                            "elapsed_s": result["elapsed_s"],
                            "error_type": result["error_type"],
                            "error_message": result["error_message"],
                            "server_log_path": model_log_path,
                        })
                        done += 1
                        if done % 10 == 0 or done == total:
                            n_failed = sum(1 for r in rows if not r["ok"])
                            print(f"Completed {done}/{total} | borked so far: {n_failed}")
        finally:
            if MANAGE_LLAMA_SERVER:
                await stop_llama_server()
                print(f"Stopped {model_cfg['label']}")

    return pd.DataFrame(rows)


In [9]:
brain_df = await run_brain_experiment(
    user_text=brain_input_text,
    model_configs=MODEL_CONFIGS,
    prompts=PROMPTS,
    n_runs=N_RUNS,
)

brain_csv = OUTPUT_DIR / "brain_results.csv"
brain_df.to_csv(brain_csv, index=False)
print("Saved:", brain_csv)
brain_df.head()


INFO:root:Starting llama-server for Small model
INFO:root:Command: llama-server -hf unsloth/Qwen3.5-9B-GGUF:UD-Q6_K_XL --no-mmproj -c 8192 --reasoning-budget 0.5 --host 127.0.0.1 --port 8080


Started Small model | pid=186136 | log=outputs/llama_server_Small_model.log
Completed 10/400 | borked so far: 0
Completed 20/400 | borked so far: 0
Completed 30/400 | borked so far: 0
Completed 40/400 | borked so far: 0
Completed 50/400 | borked so far: 0
Completed 60/400 | borked so far: 0
Completed 70/400 | borked so far: 0
Completed 80/400 | borked so far: 0
Completed 90/400 | borked so far: 0
Completed 100/400 | borked so far: 0
Completed 110/400 | borked so far: 0
Completed 120/400 | borked so far: 0
Completed 130/400 | borked so far: 0
Completed 140/400 | borked so far: 0
Completed 150/400 | borked so far: 0
Completed 160/400 | borked so far: 0
Completed 170/400 | borked so far: 0
Completed 180/400 | borked so far: 0
Completed 190/400 | borked so far: 0
Completed 200/400 | borked so far: 0


INFO:root:Starting llama-server for Large model
INFO:root:Command: llama-server -hf unsloth/Qwen3.6-35B-A3B-GGUF:UD-Q4_K_M --fit-ctx 8192 -np 1 -fa on -b 512 -ub 512 -ctk q8_0 -ctv q8_0 --temp 0.6 --top-p 0.95 --top-k 20 --min-p 0.0 --presence-penalty 0.0 --repeat-penalty 1.0 --reasoning-budget -1 --chat-template-kwargs {"preserve_thinking": true} --host 127.0.0.1 --port 8080


Stopped Small model
Started Large model | pid=189561 | log=outputs/llama_server_Large_model.log
Completed 210/400 | borked so far: 0
Completed 220/400 | borked so far: 0
Completed 230/400 | borked so far: 0
Completed 240/400 | borked so far: 0
Completed 250/400 | borked so far: 0
Completed 260/400 | borked so far: 0
Completed 270/400 | borked so far: 0
Completed 280/400 | borked so far: 0
Completed 290/400 | borked so far: 0
Completed 300/400 | borked so far: 0
Completed 310/400 | borked so far: 0
Completed 320/400 | borked so far: 0
Completed 330/400 | borked so far: 0
Completed 340/400 | borked so far: 0
Completed 350/400 | borked so far: 0
Completed 360/400 | borked so far: 0
Completed 370/400 | borked so far: 0
Completed 380/400 | borked so far: 0
Completed 390/400 | borked so far: 0
Completed 400/400 | borked so far: 0
Stopped Large model
Saved: outputs/brain_results.csv


,brain_input_text,model_label,model_name,prompt_name,run_idx,ok,attempt_count,reply,label_auto,elapsed_s,error_type,error_message,server_log_path
0,I need to wash my car. The car wash is only 10...,Small model,unsloth/Qwen3.5-9B-GGUF:UD-Q6_K_XL,baseline,1,True,1,Driving is generally the more efficient choice...,unrelated,0.534984,None,None,outputs/llama_server_Small_model.log
1,I need to wash my car. The car wash is only 10...,Small model,unsloth/Qwen3.5-9B-GGUF:UD-Q6_K_XL,baseline,2,True,1,You should drive to the car wash because 100 m...,pass,0.426763,None,None,outputs/llama_server_Small_model.log
2,I need to wash my car. The car wash is only 10...,Small model,unsloth/Qwen3.5-9B-GGUF:UD-Q6_K_XL,baseline,3,True,1,You should drive because the short distance (1...,pass,0.327566,None,None,outputs/llama_server_Small_model.log
3,I need to wash my car. The car wash is only 10...,Small model,unsloth/Qwen3.5-9B-GGUF:UD-Q6_K_XL,baseline,4,True,1,"Since the car wash is only 100 meters away, wa...",unrelated,0.539763,None,None,outputs/llama_server_Small_model.log
4,I need to wash my car. The car wash is only 10...,Small model,unsloth/Qwen3.5-9B-GGUF:UD-Q6_K_XL,baseline,5,True,1,Driving is the better option because it is fas...,unrelated,0.310759,None,None,outputs/llama_server_Small_model.log


In [10]:
failure_summary = (
    brain_df.groupby(["model_label", "prompt_name", "ok"])
    .size()
    .reset_index(name="count")
)

failure_summary["status"] = np.where(failure_summary["ok"], "success", "borked")
failure_summary["rate"] = failure_summary["count"] / N_RUNS
failure_summary


,model_label,prompt_name,ok,count,status,rate
0,Large model,baseline,True,50,success,1.0
1,Large model,novice_literal,True,50,success,1.0
2,Large model,phd_student,True,50,success,1.0
3,Large model,trick_specialist,True,50,success,1.0
4,Small model,baseline,True,50,success,1.0
5,Small model,novice_literal,True,50,success,1.0
6,Small model,phd_student,True,50,success,1.0
7,Small model,trick_specialist,True,50,success,1.0


In [11]:
failed_only = brain_df[~brain_df["ok"]]
failed_only[["model_label", "prompt_name", "run_idx", "error_type", "error_message"]].head(20)


,model_label,prompt_name,run_idx,error_type,error_message


In [12]:
summary_counts = (
    brain_df.groupby(["model_label", "prompt_name", "label_auto"])
    .size()
    .reset_index(name="count")
)

summary_rates = (
    summary_counts.assign(rate=lambda df: df["count"] / N_RUNS)
    .sort_values(["model_label", "prompt_name", "label_auto"])
)

summary_rates


,model_label,prompt_name,label_auto,count,rate
0,Large model,baseline,failed,23,0.46
1,Large model,baseline,pass,22,0.44
2,Large model,baseline,unrelated,5,0.10
3,Large model,novice_literal,failed,4,0.08
4,Large model,novice_literal,pass,44,0.88
5,Large model,novice_literal,unrelated,2,0.04
6,Large model,phd_student,failed,1,0.02
7,Large model,phd_student,pass,16,0.32
8,Large model,phd_student,unrelated,33,0.66
9,Large model,trick_specialist,pass,18,0.36


In [13]:
pass_heatmap = (
    summary_rates[summary_rates["label_auto"] == "pass"]
    .pivot(index="model_label", columns="prompt_name", values="rate")
    .fillna(0.0)
)

plt.figure(figsize=(8, 4))
plt.imshow(pass_heatmap.values, aspect="auto")
plt.colorbar(label="Pass rate")
plt.xticks(range(len(pass_heatmap.columns)), pass_heatmap.columns, rotation=30, ha="right")
plt.yticks(range(len(pass_heatmap.index)), pass_heatmap.index)
plt.title('How often did each model-prompt pair answer "driving is better"?')
plt.tight_layout()
save_path = OUTPUT_DIR / "pass_heatmap.png"
plt.savefig(save_path)
print("Saved:", save_path)

pass_heatmap


Saved: outputs/pass_heatmap.png


prompt_name,baseline,novice_literal,phd_student,trick_specialist
model_label,,,,
Large model,0.44,0.88,0.32,0.36
Small model,0.16,0.02,0.06,0.16


In [14]:
borked_heatmap = (
    failure_summary[failure_summary["status"] == "borked"]
    .pivot(index="model_label", columns="prompt_name", values="rate")
    .fillna(0.0)
)

plt.figure(figsize=(8, 4))
plt.imshow(borked_heatmap.values, aspect="auto")
plt.colorbar(label="Borked rate")
plt.xticks(range(len(borked_heatmap.columns)), borked_heatmap.columns, rotation=30, ha="right")
plt.yticks(range(len(borked_heatmap.index)), borked_heatmap.index)
plt.title("How often did each model-prompt pair bork?")
plt.tight_layout()
save_path = OUTPUT_DIR / "borked_heatmap.png"
plt.savefig(save_path)
print("Saved:", save_path)

borked_heatmap


Saved: outputs/borked_heatmap.png


/tmp/ipykernel_186070/1450497494.py:8: UserWarning: Attempting to set identical low and high xlims makes transformation singular; automatically expanding.
  plt.imshow(borked_heatmap.values, aspect="auto")
/tmp/ipykernel_186070/1450497494.py:8: UserWarning: Attempting to set identical low and high ylims makes transformation singular; automatically expanding.
  plt.imshow(borked_heatmap.values, aspect="auto")


prompt_name
model_label


In [15]:
plot_df = (
    summary_rates.pivot_table(
        index=["model_label", "prompt_name"],
        columns="label_auto",
        values="rate",
        fill_value=0.0,
    )
    .reset_index()
)

ax = plot_df.set_index(["model_label", "prompt_name"]).plot(
    kind="bar",
    stacked=True,
    figsize=(10, 5),
)
ax.set_ylabel("Rate")
ax.set_title("Pass / failed / unrelated / borked by model and prompt")
ax.legend(title="Label")
plt.tight_layout()
save_path = OUTPUT_DIR / "stacked_bar.png"
plt.savefig(save_path)
print("Saved:", save_path)


Saved: outputs/stacked_bar.png


### Optional manual relabeling

The automatic labels now follow your scoring rule:
- `pass` = reply contains **driving is better**
- `failed` = reply contains **walking is better**
- `unrelated` = any other successful reply
- `borked` = the run failed at the infrastructure level, including timeouts

The `server_log_path` column points to the llama-server log file for that model.


In [16]:
brain_df.sample(min(12, len(brain_df)), random_state=0)[
    ["model_label", "prompt_name", "run_idx", "ok", "reply", "label_auto", "error_type"]
]


,model_label,prompt_name,run_idx,ok,reply,label_auto,error_type
132,Small model,phd_student,33,True,To determine whether walking or driving to the...,unrelated,None
309,Large model,phd_student,10,True,Because the essential requirement for washing ...,unrelated,None
341,Large model,phd_student,42,True,Since the premise implicitly assumes you can t...,unrelated,None
196,Small model,trick_specialist,47,True,"The core question contains a classic ""trick"" o...",failed,None
246,Large model,baseline,47,True,"Drive it, because you need the car present at ...",unrelated,None
60,Small model,novice_literal,11,True,"Since the car wash is only 100 meters away, I ...",unrelated,None
155,Small model,trick_specialist,6,True,"**Specialist Analysis:**\n\n**Question:** ""Sho...",unrelated,None
261,Large model,novice_literal,12,True,You should drive the car there because walking...,pass,None
141,Small model,phd_student,42,True,Walking is the superior choice because the dis...,unrelated,None
214,Large model,baseline,15,True,"You should drive, since you need to transport ...",pass,None


## 5. Mouth

In [17]:
async def synthesize_tts(text, output_wav, speaker_wav=None, language=None):
    model = load_tts_model()
    speaker_wav = speaker_wav or COQUI_SPEAKER_WAV
    language = language or COQUI_LANGUAGE

    def _generate():
        model.tts_to_file(
            text=text,
            file_path=str(output_wav),
            speaker_wav=speaker_wav,
            language=language,
        )

    start = time.perf_counter()
    await asyncio.to_thread(_generate)
    elapsed = time.perf_counter() - start
    return {
        "text": text,
        "output_wav": str(output_wav),
        "speaker_wav": speaker_wav,
        "language": language,
        "elapsed_s": elapsed,
    }


In [18]:
successful_rows = brain_df[brain_df["ok"]]
mouth_text = successful_rows.iloc[0]["reply"] if len(successful_rows) else "No successful Brain reply was available."
mouth_output = OUTPUT_DIR / "mouth_output.wav"

mouth_result = await synthesize_tts(mouth_text, mouth_output)

print(json.dumps(mouth_result, indent=2))
display(Audio(filename=str(mouth_output)))


INFO:root:Loading Coqui TTS model: tts_models/multilingual/multi-dataset/xtts_v2
INFO:TTS.utils.manage:tts_models/multilingual/multi-dataset/xtts_v2 is already downloaded.
INFO:TTS.tts.models:Using model: xtts
INFO:root:Moved Coqui TTS to CUDA
INFO:TTS.utils.synthesizer:Text split into sentences.
INFO:TTS.utils.synthesizer:Input: ['Driving is generally the more efficient choice because the short distance and the time needed to wash the car would likely make the extra effort of parking and walking back unnecessary.']
INFO:TTS.utils.voices:Generated voice from reference audio
INFO:TTS.utils.synthesizer:Processing time: 1.727
INFO:TTS.utils.synthesizer:Real-time factor: 0.173


{
  "text": "Driving is generally the more efficient choice because the short distance and the time needed to wash the car would likely make the extra effort of parking and walking back unnecessary.",
  "output_wav": "outputs/mouth_output.wav",
  "speaker_wav": "/home/havoc/git_repos/Discord_AI_bot/src/outputs/mouth_input.wav",
  "language": "en",
  "elapsed_s": 1.7546903239999665
}


In [19]:
listener_template = pd.DataFrame([
    {"listener_id": 1, "resemblance_1to5": None, "naturalness_1to5": None, "clarity_1to5": None, "notes": ""},
    {"listener_id": 2, "resemblance_1to5": None, "naturalness_1to5": None, "clarity_1to5": None, "notes": ""},
    {"listener_id": 3, "resemblance_1to5": None, "naturalness_1to5": None, "clarity_1to5": None, "notes": ""},
])

listener_csv = OUTPUT_DIR / "mouth_listener_ratings_template.csv"
listener_template.to_csv(listener_csv, index=False)
listener_template


,listener_id,resemblance_1to5,naturalness_1to5,clarity_1to5,notes
0,1,None,None,None,
1,2,None,None,None,
2,3,None,None,None,


In [43]:
import math
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# -----------------------------
# 1) Load CSV
# -----------------------------
CSV_PATH = "outputs/brain_results_labeled_v2.csv"   # change if needed
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

brain_df = pd.read_csv(CSV_PATH)

# -----------------------------
# 2) Basic cleanup
# -----------------------------
brain_df["elapsed_s"] = pd.to_numeric(brain_df["elapsed_s"], errors="coerce")
brain_df["label_auto"] = brain_df["label_auto"].fillna("unrelated").astype(str).str.strip()

label_order = [
    "pass",
    "failed",
    "unrelated",
    "lack of prompt adherence",
    "loop",
]

brain_df.loc[~brain_df["label_auto"].isin(label_order), "label_auto"] = "unrelated"

# -----------------------------
# 3) Summary tables
# -----------------------------
runs_per_group = (
    brain_df.groupby(["model_label", "prompt_name"])
    .size()
    .reset_index(name="n_runs")
)

summary_counts = (
    brain_df.groupby(["model_label", "prompt_name", "label_auto"])
    .size()
    .reset_index(name="count")
)

summary_rates = summary_counts.merge(
    runs_per_group,
    on=["model_label", "prompt_name"],
    how="left"
)

summary_rates["rate"] = summary_rates["count"] / summary_rates["n_runs"]

# -----------------------------
# 4) PASS heatmap only
# -----------------------------
pass_heatmap = (
    summary_rates[summary_rates["label_auto"] == "pass"]
    .pivot(index="model_label", columns="prompt_name", values="rate")
    .fillna(0.0)
)

all_models = sorted(brain_df["model_label"].dropna().unique())
all_prompts = sorted(brain_df["prompt_name"].dropna().unique())
pass_heatmap = pass_heatmap.reindex(index=all_models, columns=all_prompts, fill_value=0.0)

plt.figure(figsize=(10, 5))
im = plt.imshow(pass_heatmap.values, aspect="auto", vmin=0.0, vmax=1.0)
plt.colorbar(im, label="Pass rate")
plt.xticks(range(len(pass_heatmap.columns)), pass_heatmap.columns, rotation=30, ha="right")
plt.yticks(range(len(pass_heatmap.index)), pass_heatmap.index)
plt.title('How often did each model-prompt pair answer "driving is better"?')
plt.tight_layout()

save_path = OUTPUT_DIR / "pass_rate_heatmap.png"
plt.savefig(save_path, bbox_inches="tight", dpi=300)
print("Saved:", save_path)
plt.close()

print("\nPass heatmap table:")
display(pass_heatmap)

# -----------------------------
# 5) Stacked bar chart with group boxes
# -----------------------------
plot_df = (
    summary_rates.pivot_table(
        index=["model_label", "prompt_name"],
        columns="label_auto",
        values="rate",
        fill_value=0.0,
    )
    .reindex(columns=label_order, fill_value=0.0)
    .reset_index()
    .sort_values(["model_label", "prompt_name"])
    .reset_index(drop=True)
)

fig, ax = plt.subplots(figsize=(20, 10))

plot_df.set_index(["model_label", "prompt_name"])[label_order].plot(
    kind="bar",
    stacked=True,
    ax=ax,
)

ax.set_ylabel("Rate")
ax.set_title("Response label distribution by model and prompt")
ax.legend(
    title="Label",
    labels=[
        "Pass",
        "Failed",
        "Unrelated",
        "Lack of prompt adherence",
        "Loop",
    ],
)

# Draw colored boxes around each model group
group_colors = {
    "Large model": "tab:red",
    "Small model": "tab:blue",
}

# If your exact labels differ, these colors will still work for whatever labels appear.
ymin, ymax = ax.get_ylim()

group_positions = (
    plot_df.reset_index()
    .groupby("model_label")["index"]
    .agg(["min", "max"])
    .reset_index()
)

for _, row in group_positions.iterrows():
    model_label = row["model_label"]
    start_idx = row["min"]
    end_idx = row["max"]

    left = start_idx - 0.5
    width = (end_idx - start_idx + 1)
    color = group_colors.get(model_label, "black")

    rect = Rectangle(
        (left, 0),
        width,
        ymax,
        fill=False,
        edgecolor=color,
        linewidth=3,
        linestyle="-",
        zorder=10,
        clip_on=False,
    )
    ax.add_patch(rect)

    # Optional model label above each box
    center_x = start_idx + (end_idx - start_idx) / 2
    ax.text(
        center_x,
        ymax * 1.02,
        model_label,
        ha="center",
        va="bottom",
        fontsize=12,
        fontweight="bold",
        color=color,
        clip_on=False,
    )

plt.tight_layout()
save_path = OUTPUT_DIR / "stacked_bar_response_labels.png"
plt.savefig(save_path, bbox_inches="tight", dpi=300)
print("Saved:", save_path)
plt.close()

display(plot_df)

# -----------------------------
# 6) Box plot of elapsed_s
# -----------------------------
brain_df["model_prompt"] = (
    brain_df["model_label"].astype(str) + " | " + brain_df["prompt_name"].astype(str)
)

elapsed_df = brain_df.dropna(subset=["elapsed_s"]).copy()

ordered_pairs = (
    elapsed_df[["model_label", "prompt_name", "model_prompt"]]
    .drop_duplicates()
    .sort_values(["model_label", "prompt_name"])
)

box_labels = ordered_pairs["model_prompt"].tolist()
box_data = [
    elapsed_df.loc[elapsed_df["model_prompt"] == label, "elapsed_s"].values
    for label in box_labels
]

plt.figure(figsize=(20, 10))
plt.boxplot(box_data, tick_labels=box_labels)
plt.xticks(rotation=30, ha="right")
plt.ylabel("elapsed_s")
plt.title("Elapsed time by model and prompt")
plt.tight_layout()

save_path = OUTPUT_DIR / "elapsed_boxplot.png"
plt.savefig(save_path, bbox_inches="tight", dpi=300)
print("Saved:", save_path)
plt.close()

# -----------------------------
# 7) Optional: separate elapsed box plots by model
# -----------------------------
models = sorted(elapsed_df["model_label"].dropna().unique())

for model in models:
    sub = elapsed_df[elapsed_df["model_label"] == model].copy()
    prompt_order = sorted(sub["prompt_name"].dropna().unique())
    data = [sub.loc[sub["prompt_name"] == p, "elapsed_s"].values for p in prompt_order]

    plt.figure(figsize=(20, 10))
    plt.boxplot(data, tick_labels=prompt_order)
    plt.xticks(rotation=30, ha="right")
    plt.ylabel("elapsed_s")
    plt.title(f"Elapsed time by prompt — {model}")
    plt.tight_layout()

    safe_model_name = str(model).replace("/", "_").replace(" ", "_")
    save_path = OUTPUT_DIR / f"elapsed_boxplot_{safe_model_name}.png"
    plt.savefig(save_path, bbox_inches="tight", dpi=300)
    print("Saved:", save_path)
    plt.close()

Saved: outputs/pass_rate_heatmap.png

Pass heatmap table:


prompt_name,baseline,novice_literal,phd_student,trick_specialist
model_label,,,,
Large model,0.48,0.92,0.92,1.00
Small model,0.26,0.40,0.04,0.26


Saved: outputs/stacked_bar_response_labels.png


label_auto,model_label,prompt_name,pass,failed,unrelated,lack of prompt adherence,loop
0,Large model,baseline,0.48,0.52,0.00,0.00,0.00
1,Large model,novice_literal,0.92,0.08,0.00,0.00,0.00
2,Large model,phd_student,0.92,0.08,0.00,0.00,0.00
3,Large model,trick_specialist,1.00,0.00,0.00,0.00,0.00
4,Small model,baseline,0.26,0.72,0.02,0.00,0.00
5,Small model,novice_literal,0.40,0.42,0.02,0.16,0.00
6,Small model,phd_student,0.04,0.70,0.22,0.02,0.02
7,Small model,trick_specialist,0.26,0.56,0.10,0.02,0.06


Saved: outputs/elapsed_boxplot.png
Saved: outputs/elapsed_boxplot_Large_model.png
Saved: outputs/elapsed_boxplot_Small_model.png


## 6. Suggested final outputs

For your short presentation, useful artifacts now include:
- one slide for **Ears** with the gold text versus the transcript
- one slide for **Brain quality** with the pass-rate heatmap
- one slide for **Brain reliability** with the borked-rate heatmap
- one quick audio demo for **Mouth**
